In [ ]:
from pathlib import Path
import json
import pandas as pd

TOPIC = "여행, 관광 및 명소"
BASE = Path("/Users/jangsehyeon/Desktop/Boostcamp/FINAL_PROJECT/012.한국어 SNS 멀티턴 대화 데이터/3.개방데이터/1.데이터")

train_dir = BASE / "Training/02.라벨링데이터"
val_dir = BASE / "Validation/02.라벨링데이터"

train_candidates = sorted([p for p in train_dir.iterdir() if p.is_dir() and p.name.startswith("TL_1.")])
val_candidates = sorted([p for p in val_dir.iterdir() if p.is_dir() and p.name.startswith("VL_1.")])

print("train_candidates:", [p.name for p in train_candidates])
print("val_candidates:", [p.name for p in val_candidates])

if len(train_candidates) != 1 or len(val_candidates) != 1:
    raise RuntimeError("후보 폴더가 1개가 아님. 폴더 선택 조건을 조정하세요.")

ROOTS = {
    "train": train_candidates[0],
    "validation": val_candidates[0],
}

def is_missing(v):
    if v is None:
        return True
    s = str(v).strip().lower()
    return s in {"", "null", "none", "nan"}

dialog_rows = []
utt_rows = []

for split, root in ROOTS.items():
    for fp in root.rglob("*.json"):
        with fp.open("r", encoding="utf-8") as f:
            d = json.load(f)

        info = d.get("info", {})
        if info.get("topic") != TOPIC:
            continue

        spk_info = info.get("speaker", {})
        if not is_missing(spk_info.get("speakerCId")):
            continue

        utterances = d.get("utterances", [])
        speakers = {u.get("speaker") for u in utterances if u.get("speaker")}
        if speakers - {"speakerA", "speakerB"}:
            continue
        if not {"speakerA", "speakerB"}.issubset(speakers):
            continue

        dialog_id = info.get("id")
        turn_cnt = len({u.get("turn_id") for u in utterances if u.get("turn_id")})

        dialog_rows.append({
            "split": split,
            "file_path": str(fp),
            "dialog_id": dialog_id,
            "topic": info.get("topic"),
            "keyword": info.get("keyword"),
            "speakerAId": spk_info.get("speakerAId"),
            "speakerBId": spk_info.get("speakerBId"),
            "speakerCId": spk_info.get("speakerCId"),
            "num_utterances": len(utterances),
            "num_turns": turn_cnt,
        })

        for i, u in enumerate(utterances):
            speaker = u.get("speaker")
            if speaker not in {"speakerA", "speakerB"}:
                continue

            slots = u.get("slot") or []
            slot_keys = [s.get("key", "") for s in slots if isinstance(s, dict)]
            slot_values = [s.get("value", "") for s in slots if isinstance(s, dict)]

            utt_rows.append({
                "split": split,
                "dialog_id": dialog_id,
                "utt_order": i,
                "turn_id": u.get("turn_id"),
                "utterance_id": u.get("utterance_id"),
                "speaker": speaker,
                "text": (u.get("text") or "").strip(),
                "speech_act": u.get("speech_act"),
                "new_word": u.get("new_word"),
                "slot_cnt": len(slots),
                "slot_keys": ";".join([k for k in slot_keys if k]),
                "slot_values": ";".join([v for v in slot_values if v]),
            })

df_dialog = pd.DataFrame(dialog_rows).sort_values(["split", "dialog_id"]).reset_index(drop=True)
df_utt = pd.DataFrame(utt_rows).sort_values(["split", "dialog_id", "utt_order"]).reset_index(drop=True)

print("ROOTS:", ROOTS)
print("df_dialog shape:", df_dialog.shape)
print("df_utt shape:", df_utt.shape)

display(df_dialog.head(3))
display(df_utt.head(5))


train_candidates: ['TL_1.일상트랜드_2.여행_관광_및_명소']
val_candidates: ['VL_1.일상트랜드_2.여행_관광_및_명소']
ROOTS: {'train': PosixPath('/Users/jangsehyeon/Desktop/Boostcamp/FINAL_PROJECT/012.한국어 SNS 멀티턴 대화 데이터/3.개방데이터/1.데이터/Training/02.라벨링데이터/TL_1.일상트랜드_2.여행_관광_및_명소'), 'validation': PosixPath('/Users/jangsehyeon/Desktop/Boostcamp/FINAL_PROJECT/012.한국어 SNS 멀티턴 대화 데이터/3.개방데이터/1.데이터/Validation/02.라벨링데이터/VL_1.일상트랜드_2.여행_관광_및_명소')}
df_dialog shape: (20658, 10)
df_utt shape: (331788, 12)


,split,file_path,dialog_id,topic,keyword,speakerAId,speakerBId,speakerCId,num_utterances,num_turns
0,train,/Users/jangsehyeon/Desktop/Boostcamp/FINAL_PRO...,000050,"여행, 관광 및 명소",2023년 울릉도 해양 체험비 지원,0001,0113,None,16,8
1,train,/Users/jangsehyeon/Desktop/Boostcamp/FINAL_PRO...,000084,"여행, 관광 및 명소",도심 한옥 스테이,0030,0113,None,16,8
2,train,/Users/jangsehyeon/Desktop/Boostcamp/FINAL_PRO...,000102,"여행, 관광 및 명소",도심 한옥 스테이,0043,0115,None,16,8


,split,dialog_id,utt_order,turn_id,utterance_id,speaker,text,speech_act,new_word,slot_cnt,slot_keys,slot_values
0,train,000050,0,000050-1,000050.1,speakerA,우리 이번 여름에 울릉도 갈까?,친교 및 잡담,,2,나라/지역/장소;날짜/기간/계절,울릉도;이번 여름
1,train,000050,1,000050-1,000050.2,speakerB,울릉도 좋지! 근데 왜 갑분 울릉도?,정보 요청,갑분,2,나라/지역/장소;평가/후기/감상,울릉도;yes
2,train,000050,2,000050-2,000050.3,speakerA,울릉도에서 해양레포츠를 돈 받고 체험할 수 있대.,정보 제공,,2,나라/지역/장소;음식/명물,울릉도;해양레포츠
3,train,000050,3,000050-2,000050.4,speakerB,오 완전 대박인데? 해양 레포츠 종류는? 다이빙도 있어?,정보 요청,대박,1,평가/후기/감상,yes
4,train,000050,4,000050-3,000050.5,speakerA,"ㅇㅇ. 스쿠버 다이빙이랑 프리다이빙, 스노클링도 있대!",정보 제공,,1,음식/명물,스쿠버 다이빙; 스노클링


In [12]:
df_utt.head(14)

,split,dialog_id,utt_order,turn_id,utterance_id,speaker,text,speech_act,new_word,slot_cnt,slot_keys,slot_values
0,train,000050,0,000050-1,000050.1,speakerA,우리 이번 여름에 울릉도 갈까?,친교 및 잡담,,2,나라/지역/장소;날짜/기간/계절,울릉도;이번 여름
1,train,000050,1,000050-1,000050.2,speakerB,울릉도 좋지! 근데 왜 갑분 울릉도?,정보 요청,갑분,2,나라/지역/장소;평가/후기/감상,울릉도;yes
2,train,000050,2,000050-2,000050.3,speakerA,울릉도에서 해양레포츠를 돈 받고 체험할 수 있대.,정보 제공,,2,나라/지역/장소;음식/명물,울릉도;해양레포츠
3,train,000050,3,000050-2,000050.4,speakerB,오 완전 대박인데? 해양 레포츠 종류는? 다이빙도 있어?,정보 요청,대박,1,평가/후기/감상,yes
4,train,000050,4,000050-3,000050.5,speakerA,"ㅇㅇ. 스쿠버 다이빙이랑 프리다이빙, 스노클링도 있대!",정보 제공,,1,음식/명물,스쿠버 다이빙; 스노클링
5,train,000050,5,000050-3,000050.6,speakerB,야 이건 무조건 가야지! 비용 전체 지원해줘?,정보 요청,,0,,
6,train,000050,6,000050-4,000050.7,speakerA,"아니.ㅋㅋ 일부 지원해주는건데, 스쿠버다이빙은 3만원이고 프리다이빙이랑 스노클링은 ...",정보 제공,,2,가격/비용;음식/명물,3만원; 1만원;스쿠버다이빙; 스노클링
7,train,000050,7,000050-4,000050.8,speakerB,"하긴, 일부라도 그게 어디야. 그만하면 완전 혜자네!",정보 제공,혜자,1,가격대,yes
8,train,000050,8,000050-5,000050.9,speakerA,그치? 근데 너 다이빙 할 수 있는 거지? ㅋㅋ,친교 및 잡담,,0,,
9,train,000050,9,000050-5,000050.10,speakerB,"야, 당근이지! 울릉도 맛집 좀 알아볼까?",친교 및 잡담,당근,1,나라/지역/장소,울릉도


In [71]:
df_utt['speaker'].value_counts()

speaker
speakerA    166000
speakerB    165788
Name: count, dtype: int64

In [13]:
import os

In [22]:
os.listdir(val_dir)

['VL_1.일상트랜드_6.콘텐츠_소비.zip',
 '.DS_Store',
 'VL_1.일상트랜드_1.건강_및_식음료.zip',
 'VL_2.시사트랜드_3.과학_기술.zip',
 'VL_1.일상트랜드_5.스포츠_및_e스포츠.zip',
 'VL_2.시사트랜드_1.정치.zip',
 'VL_1.일상트랜드_2.여행_관광_및_명소',
 'VL_1.일상트랜드_3.문화_생활_및_여가.zip',
 'VL_2.시사트랜드_2.경제_및_사회.zip',
 'VL_1.일상트랜드_2.여행_관광_및_명소.zip',
 'VL_1.일상트랜드_4.미용과_패션.zip']

In [24]:
train_fileslen = len(os.listdir(os.path.join(train_dir,'TL_1.일상트랜드_2.여행_관광_및_명소')))
valid_files_len = len(os.listdir(os.path.join(val_dir,'VL_1.일상트랜드_2.여행_관광_및_명소')))
print(f"[Total 개수]: {train_fileslen + valid_files_len}")
print(f"[Train 개수]: {train_fileslen}")
print(f"[Validation 개수]: {valid_files_len}")

[Total 개수]: 21848
[Train 개수]: 19420
[Validation 개수]: 2428


In [48]:
from pathlib import Path
import json
import pandas as pd

TOPIC = "여행, 관광 및 명소"

BASE = Path("/Users/jangsehyeon/Desktop/Boostcamp/FINAL_PROJECT/012.한국어 SNS 멀티턴 대화 데이터/3.개방데이터/1.데이터")

def pick_one_dir(parent: Path, prefix: str) -> Path:
    cands = sorted([p for p in parent.iterdir() if p.is_dir() and p.name.startswith(prefix)])
    if len(cands) != 1:
        raise RuntimeError(f"{parent} 아래 {prefix} 후보가 1개가 아님: {[c.name for c in cands]}")
    return cands[0]

ROOTS = {
    "train": pick_one_dir(BASE / "Training/02.라벨링데이터", "TL_1."),
    "validation": pick_one_dir(BASE / "Validation/02.라벨링데이터", "VL_1."),
}
print(ROOTS)

def is_missing(v):
    if v is None:
        return True
    return str(v).strip().lower() in {"", "null", "none", "nan"}

dialog_rows = []
utt_rows = []

for split, root in ROOTS.items():
    for fp in root.rglob("*.json"):
        with fp.open("r", encoding="utf-8") as f:
            d = json.load(f)

        info = d.get("info", {})
        if info.get("topic") != TOPIC:
            continue

        spk_info = info.get("speaker", {})
        if not is_missing(spk_info.get("speakerCId")):
            continue

        utterances = d.get("utterances", [])
        speakers = {u.get("speaker") for u in utterances if u.get("speaker")}
        if speakers - {"speakerA", "speakerB"}:
            continue
        if speakers != {"speakerA", "speakerB"}:
            continue

        dialog_id = info.get("id")
        dialog_rows.append({
            "split": split,
            "dialog_id": dialog_id,
            "file_path": str(fp),
            "topic": info.get("topic"),
            "keyword": info.get("keyword"),
            "speakerAId": spk_info.get("speakerAId"),
            "speakerBId": spk_info.get("speakerBId"),
            "speakerCId": spk_info.get("speakerCId"),
            "num_utterances": len(utterances),
            "num_turns": len({u.get("turn_id") for u in utterances if u.get("turn_id")}),
        })

        for i, u in enumerate(utterances):
            s = u.get("speaker")
            if s not in {"speakerA", "speakerB"}:
                continue
            slots = u.get("slot") or []
            utt_rows.append({
                "split": split,
                "dialog_id": dialog_id,
                "utt_order": i,
                "turn_id": u.get("turn_id"),
                "utterance_id": u.get("utterance_id"),
                "speaker": s,
                "text": (u.get("text") or "").strip(),
                "speech_act": (u.get("speech_act") or "").strip(),
                "new_word": (u.get("new_word") or "").strip(),
                "slot_cnt": len(slots),
            })

df_dialog = pd.DataFrame(dialog_rows).sort_values(["split", "dialog_id"]).reset_index(drop=True)
df_utt = pd.DataFrame(utt_rows).sort_values(["split", "dialog_id", "utt_order"]).reset_index(drop=True)

print("df_dialog:", df_dialog.shape)
print("df_utt:", df_utt.shape)
display(df_dialog.head(3))
display(df_utt.head(5))


{'train': PosixPath('/Users/jangsehyeon/Desktop/Boostcamp/FINAL_PROJECT/012.한국어 SNS 멀티턴 대화 데이터/3.개방데이터/1.데이터/Training/02.라벨링데이터/TL_1.일상트랜드_2.여행_관광_및_명소'), 'validation': PosixPath('/Users/jangsehyeon/Desktop/Boostcamp/FINAL_PROJECT/012.한국어 SNS 멀티턴 대화 데이터/3.개방데이터/1.데이터/Validation/02.라벨링데이터/VL_1.일상트랜드_2.여행_관광_및_명소')}
df_dialog: (20658, 10)
df_utt: (331788, 10)


,split,dialog_id,file_path,topic,keyword,speakerAId,speakerBId,speakerCId,num_utterances,num_turns
0,train,000050,/Users/jangsehyeon/Desktop/Boostcamp/FINAL_PRO...,"여행, 관광 및 명소",2023년 울릉도 해양 체험비 지원,0001,0113,None,16,8
1,train,000084,/Users/jangsehyeon/Desktop/Boostcamp/FINAL_PRO...,"여행, 관광 및 명소",도심 한옥 스테이,0030,0113,None,16,8
2,train,000102,/Users/jangsehyeon/Desktop/Boostcamp/FINAL_PRO...,"여행, 관광 및 명소",도심 한옥 스테이,0043,0115,None,16,8


,split,dialog_id,utt_order,turn_id,utterance_id,speaker,text,speech_act,new_word,slot_cnt
0,train,000050,0,000050-1,000050.1,speakerA,우리 이번 여름에 울릉도 갈까?,친교 및 잡담,,2
1,train,000050,1,000050-1,000050.2,speakerB,울릉도 좋지! 근데 왜 갑분 울릉도?,정보 요청,갑분,2
2,train,000050,2,000050-2,000050.3,speakerA,울릉도에서 해양레포츠를 돈 받고 체험할 수 있대.,정보 제공,,2
3,train,000050,3,000050-2,000050.4,speakerB,오 완전 대박인데? 해양 레포츠 종류는? 다이빙도 있어?,정보 요청,대박,1
4,train,000050,4,000050-3,000050.5,speakerA,"ㅇㅇ. 스쿠버 다이빙이랑 프리다이빙, 스노클링도 있대!",정보 제공,,1


In [49]:
K = 4
MAX_CONTEXT_CHARS = 900

pair_rows = []

for (split, dialog_id), g in df_utt.groupby(["split", "dialog_id"], sort=False):
    g = g.sort_values("utt_order")
    recs = g[["speaker", "text", "speech_act", "new_word", "utt_order"]].to_dict("records")

    for i, u in enumerate(recs):
        if u["speaker"] != "speakerB":
            continue
        if not u["text"]:
            continue

        hist = recs[max(0, i - K):i]
        if len(hist) == 0:
            continue

        lines = []
        for h in hist:
            role = "user" if h["speaker"] == "speakerA" else "assistant"
            lines.append(f"{role}: {h['text']}")

        context = "\n".join(lines)
        if len(context) > MAX_CONTEXT_CHARS:
            context = context[-MAX_CONTEXT_CHARS:]

        pair_rows.append({
            "split": split,
            "dialog_id": dialog_id,
            "target_speech_act": u["speech_act"],
            "target_has_new_word": int(bool(u["new_word"])),
            "context": context,
            "target": u["text"],
        })

df_pair = pd.DataFrame(pair_rows)
print("df_pair:", df_pair.shape)
display(df_pair.head(5))


df_pair: (165788, 6)


,split,dialog_id,target_speech_act,target_has_new_word,context,target
0,train,000050,정보 요청,1,user: 우리 이번 여름에 울릉도 갈까?,울릉도 좋지! 근데 왜 갑분 울릉도?
1,train,000050,정보 요청,1,user: 우리 이번 여름에 울릉도 갈까?\nassistant: 울릉도 좋지! 근데...,오 완전 대박인데? 해양 레포츠 종류는? 다이빙도 있어?
2,train,000050,정보 요청,0,assistant: 울릉도 좋지! 근데 왜 갑분 울릉도?\nuser: 울릉도에서 해...,야 이건 무조건 가야지! 비용 전체 지원해줘?
3,train,000050,정보 제공,1,assistant: 오 완전 대박인데? 해양 레포츠 종류는? 다이빙도 있어?\nus...,"하긴, 일부라도 그게 어디야. 그만하면 완전 혜자네!"
4,train,000050,친교 및 잡담,1,assistant: 야 이건 무조건 가야지! 비용 전체 지원해줘?\nuser: 아니...,"야, 당근이지! 울릉도 맛집 좀 알아볼까?"


In [50]:
SEED = 42
ACT_RATIO = {
    "정보 제공": 0.65,
    "친교 및 잡담": 0.25,
    "정보 요청": 0.10,
}

def sample_by_ratio(df_split: pd.DataFrame, ratio: dict, seed: int = 42):
    available = {act: int((df_split["target_speech_act"] == act).sum()) for act in ratio}
    usable = {act: r for act, r in ratio.items() if available.get(act, 0) > 0}
    if not usable:
        return df_split.iloc[0:0].copy(), {}, available

    rsum = sum(usable.values())
    usable = {act: r / rsum for act, r in usable.items()}

    scale = min(available[act] / usable[act] for act in usable)
    quota = {act: int(scale * usable[act]) for act in usable}

    sampled = []
    for act, n in quota.items():
        if n <= 0:
            continue
        g = df_split[df_split["target_speech_act"] == act]
        sampled.append(g.sample(n=n, random_state=seed, replace=False))

    out = pd.concat(sampled).sample(frac=1.0, random_state=seed).reset_index(drop=True)
    return out, quota, available

train_raw = df_pair[df_pair["split"] == "train"].reset_index(drop=True)
val_raw = df_pair[df_pair["split"] == "validation"].reset_index(drop=True)

train_bal, train_quota, train_avail = sample_by_ratio(train_raw, ACT_RATIO, seed=SEED)
val_bal, val_quota, val_avail = sample_by_ratio(val_raw, ACT_RATIO, seed=SEED)

print("train raw:", train_raw.shape, "train bal:", train_bal.shape)
print("val raw:", val_raw.shape, "val bal:", val_bal.shape)
print("train available:", train_avail)
print("train quota:", train_quota)
print(train_bal["target_speech_act"].value_counts(normalize=True).round(3))


train raw: (147357, 6) train bal: (37523, 6)
val raw: (18431, 6) val bal: (4935, 6)
train available: {'정보 제공': 133111, '친교 및 잡담': 9381, '정보 요청': 4865}
train quota: {'정보 제공': 24390, '친교 및 잡담': 9381, '정보 요청': 3752}
target_speech_act
정보 제공      0.65
친교 및 잡담    0.25
정보 요청      0.10
Name: proportion, dtype: float64


In [51]:
def build_prompt(context: str) -> str:
    return (
        "너는 친구 같은 말투의 여행 챗봇이다. "
        "자연스럽고 짧게, 과장 없이 답해.\n"
        f"{context}\nassistant:"
    )

trl_train = pd.DataFrame({
    "prompt": train_bal["context"].map(build_prompt),
    "completion": train_bal["target"],
})

trl_val = pd.DataFrame({
    "prompt": val_bal["context"].map(build_prompt),
    "completion": val_bal["target"],
})

print("trl_train:", trl_train.shape, "trl_val:", trl_val.shape)
display(trl_train.head(3))

out_dir = Path("notebooks/Jang/data")
out_dir.mkdir(parents=True, exist_ok=True)
trl_train.to_json(out_dir / "train_prompt_completion.jsonl", orient="records", lines=True, force_ascii=False)
trl_val.to_json(out_dir / "val_prompt_completion.jsonl", orient="records", lines=True, force_ascii=False)
print("saved:", out_dir)


trl_train: (37523, 2) trl_val: (4935, 2)


,prompt,completion
0,"너는 친구 같은 말투의 여행 챗봇이다. 자연스럽고 짧게, 과장 없이 답해.\nass...",와! 그럼 한국 음식도 맛있겠다! 다음에 꼭 가서 돈쭐내야지!
1,"너는 친구 같은 말투의 여행 챗봇이다. 자연스럽고 짧게, 과장 없이 답해.\nass...",나로우주센터 견학 프로그램을 중심으로 지역 대표 관광지와 전통시장에 방문하기도 한대...
2,"너는 친구 같은 말투의 여행 챗봇이다. 자연스럽고 짧게, 과장 없이 답해.\nass...",와우내 그렇게 많이 갔다니 레게노다!


saved: notebooks/Jang/data


In [65]:
print(trl_train['prompt'][20])

너는 친구 같은 말투의 여행 챗봇이다. 자연스럽고 짧게, 과장 없이 답해.
assistant: 제주도에는 풍부한 문화와 전통이 있어, 박물관이나 민속마을을 방문해보는 것도 굿굿!
user: 제주도에 가면 얼마나 오래 머물러야 할까?
assistant: 보통 4박 5일 정도가 제주도 여행하기 좋아!
user: 글쿠나, 그러면 다음 휴가 때 제주도로 떠나야지!
assistant:


In [66]:
trl_train['completion'][20]

'굿굿! 제주도 여행이 기대되네 ㅎㅎ'

In [39]:
import pandas as pd

p = trl_train["prompt"].fillna("")
c = trl_train["completion"].fillna("")

char_stats = pd.DataFrame({
    "prompt_char": p.str.len(),
    "completion_char": c.str.len(),
    "total_char": p.str.len() + c.str.len(),
})

print(char_stats.describe(percentiles=[0.5, 0.9, 0.95, 0.99]).T)
print("prompt 평균:", round(char_stats["prompt_char"].mean(), 2))
print("prompt p95:", int(char_stats["prompt_char"].quantile(0.95)))
print("total p95:", int(char_stats["total_char"].quantile(0.95)))


                   count        mean        std   min    50%    90%    95%  \
prompt_char      37523.0  182.285825  62.181300  61.0  183.0  259.0  285.0   
completion_char  37523.0   26.739626  13.403291   3.0   24.0   43.0   51.0   
total_char       37523.0  209.025451  69.240400  66.0  206.0  296.0  327.0   

                   99%    max  
prompt_char      343.0  571.0  
completion_char   73.0  217.0  
total_char       394.0  626.0  
prompt 평균: 182.29
prompt p95: 285
total p95: 327


### 의도적으로 강제함. -> assistant의 길이를 늘리는 경우

In [41]:
import numpy as np
import pandas as pd

SEED = 42

def add_buckets(df: pd.DataFrame) -> pd.DataFrame:
    x = df.copy()
    x["target_len"] = x["target"].str.len()
    x["is_info"] = x["target_speech_act"].eq("정보 제공")
    x["has_reason_word"] = x["target"].str.contains(
        r"추천|이유|좋|가성비|분위기|위치|가까|유명|인기|편해|웨이팅|주차|때문",
        regex=True,
        na=False
    )

    cond_long_info = x["is_info"] & (x["target_len"] >= 45) & x["has_reason_word"]
    cond_mid_info = x["is_info"] & (x["target_len"] >= 30)
    cond_chat = x["target_speech_act"].eq("친교 및 잡담")
    cond_ask = x["target_speech_act"].eq("정보 요청")

    x["bucket"] = np.select(
        [cond_long_info, cond_mid_info, cond_chat, cond_ask],
        ["long_info", "mid_info", "chat", "ask"],
        default="other"
    )
    return x

def ratio_sample(df: pd.DataFrame, ratio: dict, seed: int = 42) -> pd.DataFrame:
    avail = df["bucket"].value_counts().to_dict()
    use = {k: v for k, v in ratio.items() if avail.get(k, 0) > 0}
    if not use:
        return df.iloc[0:0].copy()

    s = sum(use.values())
    use = {k: v / s for k, v in use.items()}

    scale = min(avail[k] / use[k] for k in use)
    quota = {k: int(scale * use[k]) for k in use}

    sampled = []
    for k, n in quota.items():
        g = df[df["bucket"] == k]
        sampled.append(g.sample(n=n, random_state=seed, replace=False))

    out = pd.concat(sampled).sample(frac=1.0, random_state=seed).reset_index(drop=True)
    print("available:", avail)
    print("quota:", quota)
    print("final ratio:", out["bucket"].value_counts(normalize=True).round(3).to_dict())
    return out

# 1) source
train_src = add_buckets(df_pair[df_pair["split"] == "train"])
val_src = add_buckets(df_pair[df_pair["split"] == "validation"])

# 2) mix (정보형 강화)
TARGET_RATIO = {
    "long_info": 0.45,
    "mid_info": 0.25,
    "chat": 0.20,
    "ask": 0.10,
}
train_bal = ratio_sample(train_src, TARGET_RATIO, seed=SEED)
val_bal = ratio_sample(val_src, TARGET_RATIO, seed=SEED)

# 3) TRL용 prompt/completion
def build_prompt(context: str) -> str:
    return (
        "너는 친구 같은 말투의 여행 챗봇이다. "
        "질문이 추천/정보 요청이면 2~3개 선택지와 이유를 자연스럽게 포함해 답해.\n"
        f"{context}\nassistant:"
    )

trl_train = pd.DataFrame({
    "prompt": train_bal["context"].map(build_prompt),
    "completion": train_bal["target"]
})
trl_val = pd.DataFrame({
    "prompt": val_bal["context"].map(build_prompt),
    "completion": val_bal["target"]
})

print(trl_train.shape, trl_val.shape)
print(trl_train["completion"].str.len().describe(percentiles=[0.5, 0.9, 0.95]))


available: {'other': 98927, 'mid_info': 32545, 'chat': 9381, 'ask': 4865, 'long_info': 1639}
quota: {'long_info': 1639, 'mid_info': 910, 'chat': 728, 'ask': 364}
final ratio: {'long_info': 0.45, 'mid_info': 0.25, 'chat': 0.2, 'ask': 0.1}
available: {'other': 12187, 'mid_info': 4063, 'chat': 1234, 'ask': 730, 'long_info': 217}
quota: {'long_info': 217, 'mid_info': 120, 'chat': 96, 'ask': 48}
final ratio: {'long_info': 0.451, 'mid_info': 0.249, 'chat': 0.2, 'ask': 0.1}
(3641, 2) (481, 2)
count    3641.000000
mean       46.539687
std        20.996667
min         9.000000
50%        45.000000
90%        71.000000
95%        86.000000
max       186.000000
Name: completion, dtype: float64


In [42]:
trl_train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3641 entries, 0 to 3640
Data columns (total 2 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   prompt      3641 non-null   object
 1   completion  3641 non-null   object
dtypes: object(2)
memory usage: 57.0+ KB


In [43]:
trl_train.head()

,prompt,completion
0,너는 친구 같은 말투의 여행 챗봇이다. 질문이 추천/정보 요청이면 2~3개 선택지와...,마경대에 대한 좋은 의견과 아이디어가 있다면 의견을 남겨봐. 발전에 도움이 될 거야!
1,너는 친구 같은 말투의 여행 챗봇이다. 질문이 추천/정보 요청이면 2~3개 선택지와...,여름 좋지. 그리구 해산물이 맛있으니까 꼭 먹어봐.
2,너는 친구 같은 말투의 여행 챗봇이다. 질문이 추천/정보 요청이면 2~3개 선택지와...,과거와 현대가 어우러진 북촌에서 레알 다양한 경험을 할 수 있겠다
3,너는 친구 같은 말투의 여행 챗봇이다. 질문이 추천/정보 요청이면 2~3개 선택지와...,좋긴한데 비즈니스석 아마 다 마감일껄? 요즘 일반석보다 비즈니스석 마감이 빠르대.
4,너는 친구 같은 말투의 여행 챗봇이다. 질문이 추천/정보 요청이면 2~3개 선택지와...,"물놀이로 시원한 첨벙섬, 푸른 여름 밤하늘을 보기 좋은 캠핑섬, 뜨거운 여름바다의 ..."


In [47]:
for a,b in zip(trl_train['prompt'][:5].tolist(),trl_train['completion'][:5].tolist()):
    print(a)
    print(b)

너는 친구 같은 말투의 여행 챗봇이다. 질문이 추천/정보 요청이면 2~3개 선택지와 이유를 자연스럽게 포함해 답해.
assistant: 맞아! 관광지 입장권과 기념품도 받을 수 있어. 최고야!
user: 글쿠나! 그리고 마경대의 역사에 대해 알려주는 화상 인터뷰도 있나봐.
assistant: 맞아! 전문가와 학자들이 마경대의 고사를 들려줄 거야. 레알 흥미로워!
user: 와우내 그럼 레알 모습을 알 수 있겠네!
assistant:
마경대에 대한 좋은 의견과 아이디어가 있다면 의견을 남겨봐. 발전에 도움이 될 거야!
너는 친구 같은 말투의 여행 챗봇이다. 질문이 추천/정보 요청이면 2~3개 선택지와 이유를 자연스럽게 포함해 답해.
assistant: 항공권은 언제 타느냐에 따라 가격이 차이가 넘 심해서.. 나는 편도에 90만원 정도 줬어.
user: 아. 그렇긴 하지. 나도 평일에 갈라구 ㅎㅎ
assistant: 언제쯤 갈 생각인데? 스페인은 바다도 예쁜곳이 많아서 따듯할 때 가면 더 좋아.
user: 글쎄. 언제 갈지는 아직 정확히 안 정해서. 여름도 좋겠당.
assistant:
여름 좋지. 그리구 해산물이 맛있으니까 꼭 먹어봐.
너는 친구 같은 말투의 여행 챗봇이다. 질문이 추천/정보 요청이면 2~3개 선택지와 이유를 자연스럽게 포함해 답해.
assistant: 와 전통공예 체험 해보고 싶다
user: 그리고 아트 로드도 있어
assistant: 아트 로드에서는 뭐 볼 수 있어?
user: 전통 미술 전시를 볼 수 있어
assistant:
과거와 현대가 어우러진 북촌에서 레알 다양한 경험을 할 수 있겠다
너는 친구 같은 말투의 여행 챗봇이다. 질문이 추천/정보 요청이면 2~3개 선택지와 이유를 자연스럽게 포함해 답해.
user: 야 우리 영국 여행 가기로 한 거 비즈니스 탈까? 시간 엄청 길던데.
assistant:
좋긴한데 비즈니스석 아마 다 마감일껄? 요즘 일반석보다 비즈니스석 마감이 빠르대.
너는 친구 같은 말투의 여행 챗봇이다. 질문이 추천/정보 요

In [ ]:
# 턴 1
"system": "너는 친구 같은 말투의 여행 챗봇이다. 질문이 추천/정보 요청이면 2~3개 선택지와 이유를 자연스럽게 포함해 답해."
"user": "안녕?? 부산 맛집 추천 ㄱㄱ"
"assistant": 생성

In [ ]:
# 턴 2
"system": "너는 친구 같은 말투의 여행 챗봇이다. 질문이 추천/정보 요청이면 2~3개 선택지와 이유를 자연스럽게 포함해 답해."
"user": "안녕?? 부산 맛집 추천 ㄱㄱ"
"assistant": "나였으면 우리집 추천함"
"user": "너 부산 안살잖아." # 이것만 내 챗봇에서 input으로 넣음
"assistant": "장난임. 지석국밥 추천. 여기 맛있음"


In [ ]:
# 턴 2
"system": "너는 친구 같은 말투의 여행 챗봇이다. 질문이 추천/정보 요청이면 2~3개 선택지와 이유를 자연스럽게 포함해 답해."
"user": "안녕?? 부산 맛집 추천 ㄱㄱ"
"assistant": "나였으면 우리집 추천함"
"user": "너 부산 안살잖아." 
"assistant": "장난임. 지석국밥 추천. 여기 맛있음"
"user": "거기는 영업시간이 어떻게 되는데?" # 이것만 내 챗봇에서 input으로 넣음

In [ ]:
# single turn
"user": "assistant"
"정보 요청": "정보 제공"

# "정보 제공" 길이가 긴것 필터링
# "정보 요청"의 keyword가 [] 안에 있는것. 

In [73]:
from pathlib import Path
import json
import re
import pandas as pd

# =========================
# 0) Config
# =========================
TOPIC = "여행, 관광 및 명소"
BASE = Path("/Users/jangsehyeon/Desktop/Boostcamp/FINAL_PROJECT/012.한국어 SNS 멀티턴 대화 데이터/3.개방데이터/1.데이터")

K_TURN = 3
MAX_CONTEXT_CHARS = 1200
SEED = 42

# core 조건
MIN_TARGET_CHARS = 25
MIN_QUALITY_SCORE = 1

# style augment 비율 (core 대비)
CHAT_AUG_RATIO = 0.30
ASK_AUG_RATIO = 0.15

# 정보성 표현(소프트 품질 신호)
REASON_RE = re.compile(r"추천|이유|좋|가성비|분위기|위치|가까|유명|인기|편해|동선|코스|웨이팅|주차|포인트")

SYSTEM_PROMPT = (
    "너는 친구 같은 말투의 여행 챗봇이다. "
    "질문이 추천/정보 요청이면 2~3개 선택지와 짧은 이유를 자연스럽게 말해."
)

OUT_DIR = Path("notebooks/Jang/data")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# =========================
# 1) Helper
# =========================
def pick_one_dir(parent: Path, prefix: str) -> Path:
    cands = sorted([p for p in parent.iterdir() if p.is_dir() and p.name.startswith(prefix)])
    if len(cands) != 1:
        raise RuntimeError(f"{parent} 아래 {prefix} 후보가 1개가 아님: {[c.name for c in cands]}")
    return cands[0]

def is_missing(v):
    if v is None:
        return True
    return str(v).strip().lower() in {"", "null", "none", "nan"}

def build_context(pairs, idx, k_turn=3, max_chars=1200):
    # 이전 완료 턴들 + 현재 user(A) 질문으로 context 구성
    lines = []
    start = max(0, idx - k_turn)
    for j in range(start, idx):
        lines.append(f"user: {pairs[j]['a_text']}")
        lines.append(f"assistant: {pairs[j]['b_text']}")
    lines.append(f"user: {pairs[idx]['a_text']}")
    ctx = "\n".join(lines)
    if len(ctx) > max_chars:
        ctx = ctx[-max_chars:]
    return ctx

def extract_pairs_from_dialog(d):
    info = d.get("info", {})
    spk_info = info.get("speaker", {})

    # 파일(대화) 필터
    if info.get("topic") != TOPIC:
        return []
    if not is_missing(spk_info.get("speakerCId")):
        return []

    # A/B 발화만 정리
    utts = []
    for i, u in enumerate(d.get("utterances", [])):
        s = u.get("speaker")
        if s not in {"speakerA", "speakerB"}:
            continue
        slot = u.get("slot") or []
        utts.append({
            "utt_order": i,
            "speaker": s,
            "text": (u.get("text") or "").strip(),
            "speech_act": (u.get("speech_act") or "").strip(),
            "new_word": (u.get("new_word") or "").strip(),
            "slot_cnt": len(slot),
            "turn_id": u.get("turn_id"),
            "utterance_id": u.get("utterance_id"),
        })

    speakers = {u["speaker"] for u in utts}
    if speakers != {"speakerA", "speakerB"}:
        return []

    # 턴 단위 A->B pair 추출
    pairs = []
    i = 0
    while i < len(utts) - 1:
        a = utts[i]
        b = utts[i + 1]
        if a["speaker"] == "speakerA" and b["speaker"] == "speakerB":
            same_turn = True
            if a.get("turn_id") and b.get("turn_id"):
                same_turn = (a["turn_id"] == b["turn_id"])
            if same_turn and a["text"] and b["text"]:
                pairs.append({
                    "dialog_id": info.get("id"),
                    "keyword": info.get("keyword"),
                    "topic": info.get("topic"),
                    "speakerAAge": spk_info.get("speakerAAge"),
                    "speakerBAge": spk_info.get("speakerBAge"),

                    "a_text": a["text"],
                    "a_speech_act": a["speech_act"],
                    "a_slot_cnt": a["slot_cnt"],
                    "a_turn_id": a["turn_id"],
                    "a_utterance_id": a["utterance_id"],

                    "b_text": b["text"],
                    "b_speech_act": b["speech_act"],
                    "b_slot_cnt": b["slot_cnt"],
                    "b_new_word": b["new_word"],
                    "b_turn_id": b["turn_id"],
                    "b_utterance_id": b["utterance_id"],
                })
                i += 2
                continue
        i += 1

    return pairs

def add_quality_columns(df):
    x = df.copy()
    x["target_len"] = x["b_text"].str.len()
    x["has_reason_word"] = x["b_text"].str.contains(REASON_RE, na=False).astype(int)
    x["has_slot"] = (x["b_slot_cnt"] > 0).astype(int)

    # 핵심 학습 타깃
    x["is_core"] = (
        (x["a_speech_act"] == "정보 요청") &
        (x["b_speech_act"] == "정보 제공") &
        (x["target_len"] >= MIN_TARGET_CHARS)
    )

    # 길이/근거표현/slot 기반 소프트 품질 점수
    x["quality_score"] = (
        (x["target_len"] >= 45).astype(int) +
        x["has_reason_word"] +
        x["has_slot"]
    )
    return x

def make_split_dataset(df_split):
    core = df_split[df_split["is_core"] & (df_split["quality_score"] >= MIN_QUALITY_SCORE)].copy()

    # 친구 톤 유지를 위해 일부 비-core 보강
    chat_pool = df_split[(df_split["b_speech_act"] == "친교 및 잡담") & (df_split["target_len"] >= 8)].copy()
    ask_pool = df_split[(df_split["b_speech_act"] == "정보 요청") & (df_split["target_len"] >= 8)].copy()

    chat_n = min(len(chat_pool), int(len(core) * CHAT_AUG_RATIO))
    ask_n = min(len(ask_pool), int(len(core) * ASK_AUG_RATIO))

    chat_add = chat_pool.sample(n=chat_n, random_state=SEED) if chat_n > 0 else chat_pool.iloc[0:0]
    ask_add = ask_pool.sample(n=ask_n, random_state=SEED) if ask_n > 0 else ask_pool.iloc[0:0]

    out = pd.concat([core, chat_add, ask_add], axis=0)
    out = out.drop_duplicates(subset=["dialog_id", "b_utterance_id"])
    out = out.sample(frac=1.0, random_state=SEED).reset_index(drop=True)
    return out

# =========================
# 2) Extract pair_df
# =========================
ROOTS = {
    "train": pick_one_dir(BASE / "Training/02.라벨링데이터", "TL_1."),
    "validation": pick_one_dir(BASE / "Validation/02.라벨링데이터", "VL_1."),
}
print("ROOTS:", ROOTS)

rows = []
for split, root in ROOTS.items():
    for fp in root.rglob("*.json"):
        with fp.open("r", encoding="utf-8") as f:
            d = json.load(f)

        pairs = extract_pairs_from_dialog(d)
        if not pairs:
            continue

        for idx, p in enumerate(pairs):
            row = dict(p)
            row["split"] = split
            row["pair_idx"] = idx
            row["context"] = build_context(pairs, idx, k_turn=K_TURN, max_chars=MAX_CONTEXT_CHARS)
            rows.append(row)

pair_df = pd.DataFrame(rows)
pair_df = add_quality_columns(pair_df)

print("pair_df:", pair_df.shape)
display(pair_df.head(3))

# =========================
# 3) Build final train/val
# =========================
train_raw = pair_df[pair_df["split"] == "train"].reset_index(drop=True)
val_raw = pair_df[pair_df["split"] == "validation"].reset_index(drop=True)

train_final = make_split_dataset(train_raw)
val_final = make_split_dataset(val_raw)

print("train_raw:", train_raw.shape, "val_raw:", val_raw.shape)
print("train_final:", train_final.shape, "val_final:", val_final.shape)
print("train_final b_speech_act:", train_final["b_speech_act"].value_counts().to_dict())
print("train_final target_len:")
print(train_final["target_len"].describe(percentiles=[0.5, 0.9, 0.95, 0.99]))

# =========================
# 4) TRL format
# =========================
trl_train = pd.DataFrame({
    "prompt": train_final["context"].map(lambda c: f"{SYSTEM_PROMPT}\n{c}\nassistant:"),
    "completion": train_final["b_text"],
})
trl_val = pd.DataFrame({
    "prompt": val_final["context"].map(lambda c: f"{SYSTEM_PROMPT}\n{c}\nassistant:"),
    "completion": val_final["b_text"],
})

print("trl_train:", trl_train.shape, "trl_val:", trl_val.shape)
print("prompt len stats:")
print(trl_train["prompt"].str.len().describe(percentiles=[0.5, 0.9, 0.95, 0.99]))
print("completion len stats:")
print(trl_train["completion"].str.len().describe(percentiles=[0.5, 0.9, 0.95, 0.99]))

# 필요하면 저장
# trl_train.to_json(OUT_DIR / "train_prompt_completion.jsonl", orient="records", lines=True, force_ascii=False)
# trl_val.to_json(OUT_DIR / "val_prompt_completion.jsonl", orient="records", lines=True, force_ascii=False)
# print("saved to:", OUT_DIR)


ROOTS: {'train': PosixPath('/Users/jangsehyeon/Desktop/Boostcamp/FINAL_PROJECT/012.한국어 SNS 멀티턴 대화 데이터/3.개방데이터/1.데이터/Training/02.라벨링데이터/TL_1.일상트랜드_2.여행_관광_및_명소'), 'validation': PosixPath('/Users/jangsehyeon/Desktop/Boostcamp/FINAL_PROJECT/012.한국어 SNS 멀티턴 대화 데이터/3.개방데이터/1.데이터/Validation/02.라벨링데이터/VL_1.일상트랜드_2.여행_관광_및_명소')}
pair_df: (165786, 24)


,dialog_id,keyword,topic,speakerAAge,speakerBAge,a_text,a_speech_act,a_slot_cnt,a_turn_id,a_utterance_id,...,b_turn_id,b_utterance_id,split,pair_idx,context,target_len,has_reason_word,has_slot,is_core,quality_score
0,228894,숙박,"여행, 관광 및 명소",10,20,와! 작년에 국내 여행객 중 3명 중 1명이 호텔을 이용한대!,정보 제공,2,228894-1,228894.1,...,228894-1,228894.2,train,0,user: 와! 작년에 국내 여행객 중 3명 중 1명이 호텔을 이용한대!,26,0,1,False,1
1,228894,숙박,"여행, 관광 및 명소",10,20,"숙박비도 늘었대! 하루에 6만 9,000원이나 된대ㅋㅋ",정보 제공,1,228894-2,228894.3,...,228894-2,228894.4,train,1,user: 와! 작년에 국내 여행객 중 3명 중 1명이 호텔을 이용한대!\nassi...,44,0,1,False,1
2,228894,숙박,"여행, 관광 및 명소",10,20,응! 그래! 전체 여행비에서 숙박비 비중이 계속 커지고 있대!,정보 제공,0,228894-3,228894.5,...,228894-3,228894.6,train,2,user: 와! 작년에 국내 여행객 중 3명 중 1명이 호텔을 이용한대!\nassi...,29,0,1,False,1


train_raw: (147355, 24) val_raw: (18431, 24)
train_final: (11326, 24) val_final: (1450, 24)
train_final b_speech_act: {'정보 제공': 7812, '친교 및 잡담': 2343, '정보 요청': 1171}
train_final target_len:
count    11326.000000
mean        44.573812
std         21.690774
min          8.000000
50%         40.000000
90%         72.000000
95%         87.000000
99%        117.000000
max        217.000000
Name: target_len, dtype: float64
trl_train: (11326, 2) trl_val: (1450, 2)
prompt len stats:
count    11326.000000
mean       334.061893
std        117.884668
min         88.000000
50%        344.000000
90%        475.000000
95%        515.000000
99%        614.000000
max        799.000000
Name: prompt, dtype: float64
completion len stats:
count    11326.000000
mean        44.573812
std         21.690774
min          8.000000
50%         40.000000
90%         72.000000
95%         87.000000
99%        117.000000
max        217.000000
Name: completion, dtype: float64


In [88]:
trl_train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11326 entries, 0 to 11325
Data columns (total 2 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   prompt      11326 non-null  object
 1   completion  11326 non-null  object
dtypes: object(2)
memory usage: 177.1+ KB


In [77]:
print(trl_train['prompt'][0])

너는 친구 같은 말투의 여행 챗봇이다. 질문이 추천/정보 요청이면 2~3개 선택지와 짧은 이유를 자연스럽게 말해.
user: 두근세근~ 벌써 떨림ㅎㅎ. 일주일 동안이면 그동안 안 가본 곳 위주로 가자.
assistant: ㅋㅋ 사우디아라비아 어때? 일주일 여행 상품 출시 됐던데.
user: 와~ 사우디아라비아 여행이 일주일에 가능함?
assistant: 여행사 상품 보니까 3박 6일도 있고 4박 7일도 가능함. 엄청 다양해서 우리 충분히 갔다 올 수 있음.
user: 신박하기는 하네 ㅎㅎ. 사우디아라비아까지 비행기로 얼마나 걸림?
assistant: 일단 인천에서 리야드까지 편도 12시간 40분 정도 소요되는 듯. 그래도 이번이 좋은 기회니까 가보자.
user: 사우디아라비아로 여행 가기 전에 꼭 알아야 하는 것 좀 알려줘.
assistant:


In [78]:
print(trl_train['completion'][0])

뭐 이슬람 국가니까 주류랑 돼지고기 금지, 히잡 같은 거 착용해야겠징 ㅎㅎ 재미있을 듯함.


In [93]:
for idx, row in trl_train.sample(n=10, random_state=42).iterrows():
    p = row['prompt']
    c = row['completion']
    print("====="*60)
    print(f"[Index]: {idx}")
    print(f"[Prompt]: {p}.") # 너무 길면 잘라서 보기
    print(f"[Completion]: {c}")

[Index]: 1250
[Prompt]: 너는 친구 같은 말투의 여행 챗봇이다. 질문이 추천/정보 요청이면 2~3개 선택지와 짧은 이유를 자연스럽게 말해.
user: 여행천재라니 급 땡기는데 이벤트는 어떤 내용이야?
assistant: 여행치트키 미션이 4개 있는데 미션을 완료하면 1등을 추첨해서 1000만원 상당의 세계여행 티켓을 준대.
user: 1000만원짜리 세계여행 티켓이라니 진짜 두쿵한다.
assistant: ㅎㅎ 1등하면 레알 갬동받아서 쓰러질지도 몰라.
user: 어떤 미션을 해야 되는거야?
assistant: 첫번째는 하나투어앱 설치하는 거고 설치만 하도 모두한테 광동제약 비타500을 준대.
user: 하나투어가 아주 통 크게 쏘네? 다른 미션들은 어떤 건지 궁금하당.
assistant:.
[Completion]: 두번째는 하나라이브 여행특가 알람 신청하기고 세번째는 플레너에 원하는 여행일정 담기. 그리고 마지막 미션은 여행만보 서비스 이벤트 응모하기야.
[Index]: 2816
[Prompt]: 너는 친구 같은 말투의 여행 챗봇이다. 질문이 추천/정보 요청이면 2~3개 선택지와 짧은 이유를 자연스럽게 말해.
user: 키자니아 GO! 그게 뭐야?? 그냥 평범한 키자니아하고 다른 거야?
assistant: 응! 키자니아는 서울하고 부산에만 있지만, 키자니아 GO는 일정한 장소나 지역에서 진행하는 행사야.
user: 키자니아가 열심히 하네...근데 그 프로그램이 특정 장소에서 하면 직업 체험 종류 줄어들지 않아?
assistant: 응응. 전체 테마파크를 옮길 순 없으니까ㅋㅋ. 최대로 11개의 체험관 설치할 수 있다고 하더라고!
user: 그래? 그럼 어떤 체험들을 할 수 있는 거야? 설치되는 곳마다 다 다른 건가?
assistant: 의료센터, 항공 승무원 훈련센터, CSI, 소방관 체험 등이 가능해. 그리고 새로운 체험 시설도 만들 수 있다고!
user: 어린애들한테 정말 좋겠다 ㅋㅋ. 그럼 대구에서 어딘가에서 하나봐?
assistant:.

In [98]:
from pathlib import Path
import json
import pandas as pd

def pick_one_dir(parent: Path, prefix: str) -> Path:
    cands = sorted([p for p in parent.iterdir() if p.is_dir() and p.name.startswith(prefix)])
    if len(cands) != 1:
        raise RuntimeError(f"{parent} 아래 {prefix} 후보가 1개가 아님: {[c.name for c in cands]}")
    return cands[0]

def is_missing(v):
    return v is None or str(v).strip().lower() in {"", "null", "none", "nan"}

def build_context_from_pairs(pairs, idx, k_turn=3, max_chars=1200):
    lines = []
    start = max(0, idx - k_turn)
    for j in range(start, idx):
        lines.append(f"user: {pairs[j]['a_text']}")
        lines.append(f"assistant: {pairs[j]['b_text']}")
    lines.append(f"user: {pairs[idx]['a_text']}")
    ctx = "\n".join(lines)
    if len(ctx) > max_chars:
        ctx = ctx[-max_chars:]
    return ctx

def extract_ab_pairs(dialog_obj, topic=TOPIC):
    info = dialog_obj.get("info", {})
    spk = info.get("speaker", {})

    if info.get("topic") != topic:
        return []
    if not is_missing(spk.get("speakerCId")):
        return []

    utts = []
    for i, u in enumerate(dialog_obj.get("utterances", [])):
        s = u.get("speaker")
        if s not in {"speakerA", "speakerB"}:
            continue
        utts.append({
            "speaker": s,
            "text": (u.get("text") or "").strip(),
            "speech_act": (u.get("speech_act") or "").strip(),
            "slot_cnt": len(u.get("slot") or []),
            "turn_id": u.get("turn_id"),
            "utterance_id": u.get("utterance_id"),
            "utt_order": i,
        })

    speakers = {u["speaker"] for u in utts}
    if speakers != {"speakerA", "speakerB"}:
        return []

    pairs = []
    i = 0
    while i < len(utts) - 1:
        a = utts[i]
        b = utts[i + 1]
        if a["speaker"] == "speakerA" and b["speaker"] == "speakerB":
            same_turn = True
            if a.get("turn_id") and b.get("turn_id"):
                same_turn = (a["turn_id"] == b["turn_id"])
            if same_turn and a["text"] and b["text"]:
                pairs.append({
                    "dialog_id": info.get("id"),
                    "keyword": info.get("keyword"),
                    "a_text": a["text"],
                    "a_speech_act": a["speech_act"],
                    "a_slot_cnt": a["slot_cnt"],
                    "a_turn_id": a["turn_id"],
                    "a_utterance_id": a["utterance_id"],
                    "b_text": b["text"],
                    "b_speech_act": b["speech_act"],
                    "b_slot_cnt": b["slot_cnt"],
                    "b_turn_id": b["turn_id"],
                    "b_utterance_id": b["utterance_id"],
                })
                i += 2
                continue
        i += 1

    return pairs

def collect_pair_samples(roots, topic=TOPIC, k_turn=3, max_context_chars=1200):
    rows = []
    for split, root in roots.items():
        for fp in root.rglob("*.json"):
            with fp.open("r", encoding="utf-8") as f:
                d = json.load(f)

            pairs = extract_ab_pairs(d, topic=topic)
            if not pairs:
                continue

            for idx, p in enumerate(pairs):
                row = dict(p)
                row["split"] = split
                row["pair_idx"] = idx
                row["context"] = build_context_from_pairs(pairs, idx, k_turn=k_turn, max_chars=max_context_chars)
                row["target"] = p["b_text"]
                row["target_len"] = len(p["b_text"])
                rows.append(row)

    return pd.DataFrame(rows)

# 1) core 함수: A=정보 요청, B=정보 제공 쌍만
def make_core_req_info_dataset(roots, topic=TOPIC, k_turn=3, max_context_chars=1200, min_target_chars=25):
    pair_df = collect_pair_samples(roots, topic=topic, k_turn=k_turn, max_context_chars=max_context_chars)
    core_df = pair_df[
        (pair_df["a_speech_act"] == "정보 요청") &
        (pair_df["b_speech_act"] == "정보 제공") &
        (pair_df["target_len"] >= min_target_chars)
    ].copy().reset_index(drop=True)
    return core_df, pair_df

def _add_bucket(df):
    x = df.copy()
    x["bucket"] = "other"
    x.loc[(x["a_speech_act"] == "정보 요청") & (x["b_speech_act"] == "정보 제공"), "bucket"] = "core_req_info"
    x.loc[x["b_speech_act"] == "친교 및 잡담", "bucket"] = "chat"
    x.loc[x["b_speech_act"] == "정보 요청", "bucket"] = "ask_back"
    return x

def _sample_by_ratio(df, ratio, seed=42):
    avail = df["bucket"].value_counts().to_dict()
    use = {k: v for k, v in ratio.items() if avail.get(k, 0) > 0}
    if not use:
        return df.iloc[0:0].copy(), {}, avail

    s = sum(use.values())
    use = {k: v / s for k, v in use.items()}

    scale = min(avail[k] / use[k] for k in use)
    quota = {k: int(scale * use[k]) for k in use}

    parts = []
    for k, n in quota.items():
        if n <= 0:
            continue
        g = df[df["bucket"] == k]
        parts.append(g.sample(n=n, random_state=seed, replace=False))

    out = pd.concat(parts).sample(frac=1.0, random_state=seed).reset_index(drop=True)
    return out, quota, avail

def make_ratio_mixed_dataset(pair_df, ratio=None, min_chars_by_bucket=None, seed=42):
    if ratio is None:
        ratio = {"core_req_info": 0.70, "chat": 0.20, "ask_back": 0.10}
    if min_chars_by_bucket is None:
        min_chars_by_bucket = {"core_req_info": 25, "chat": 8, "ask_back": 8, "other": 8}

    x = _add_bucket(pair_df)

    keep = pd.Series(True, index=x.index)
    for b, min_len in min_chars_by_bucket.items():
        keep &= ~((x["bucket"] == b) & (x["target_len"] < min_len))
    x = x[keep].copy()

    train = x[x["split"] == "train"].copy()
    val = x[x["split"] == "validation"].copy()

    train_mix, train_quota, train_avail = _sample_by_ratio(train, ratio, seed=seed)
    val_mix, val_quota, val_avail = _sample_by_ratio(val, ratio, seed=seed)

    debug = {
        "train_avail": train_avail,
        "train_quota": train_quota,
        "val_avail": val_avail,
        "val_quota": val_quota,
    }
    return train_mix, val_mix, debug


In [118]:
BASE = Path("/Users/jangsehyeon/Desktop/Boostcamp/FINAL_PROJECT/012.한국어 SNS 멀티턴 대화 데이터/3.개방데이터/1.데이터")
ROOTS = {
    "train": pick_one_dir(BASE / "Training/02.라벨링데이터", "TL_1."),
    "validation": pick_one_dir(BASE / "Validation/02.라벨링데이터", "VL_1."),
}

core_df, pair_df = make_core_req_info_dataset(ROOTS, k_turn=3, min_target_chars=35)
train_mix, val_mix, debug = make_ratio_mixed_dataset(
    pair_df,
    # core_reg_info: A=정보 요청, B=정보 제공
    # chat: B가 친교 및 잡담
    # ask_back: B가 정보 요청
    ratio={"core_req_info": 0.70, "chat": 0.20, "ask_back": 0.10},
    min_chars_by_bucket={"core_req_info": 35, "chat": 20, "ask_back": 20, "other": 20},
)

print(core_df.shape, pair_df.shape)
print(train_mix.shape, val_mix.shape)
print(debug)

(7053, 17) (165786, 17)
(8970, 18) (1105, 18)
{'train_avail': {'other': 71768, 'chat': 7256, 'core_req_info': 6279, 'ask_back': 4072}, 'train_quota': {'core_req_info': 6279, 'chat': 1794, 'ask_back': 897}, 'val_avail': {'other': 9059, 'chat': 913, 'core_req_info': 774, 'ask_back': 612}, 'val_quota': {'core_req_info': 774, 'chat': 221, 'ask_back': 110}}


In [119]:
for idx, row in train_mix.sample(n=10, random_state=42).iterrows():
    p = row['context']
    c = row['target']
    b = row['bucket']
    print("====="*60)
    print(f"[Index]: {idx}")
    print(f"[Type]: {b}")
    print(f"[Context]: {p}.")
    print(f"[Target]: {c}")

[Index]: 8348
[Type]: core_req_info
[Context]: user: 뭔솔? 강릉 시티버스가 뭐야?
assistant: 강릉 커피 유명한건 알지? 시티버스는 커피랑 바다를 제대로 즐길 수 있는 최상의 대중교통이란다
user: 한 손에 커피들고 여행이라니 완전 내 서타일이네. 근데 언제부터 운행된거야?
assistant: 6월1일부터 시작됐대. 요금은 시내버스지만 코스는 리무진이라고 홍보하더라.
user: 난 쟈철 아니면 뚜벅이라...그래서 시티버스 요금이 얼마임?
assistant: 1,700원이고 환승도 되니까 짱이지?
user: 시티버스면 버정 있겠네. 어디서 어디까지 가는거야?.
[Target]: 안목 커피 거리 정류장에서 주문진 해변까지 편도 23키로 해안을 달린다니 볼 곳 많겠지?
[Index]: 2034
[Type]: core_req_info
[Context]: user: 요즘 몽골 여행이 핫해졌다던데 들어봤어?
assistant: 나 혼자 산다 10주년 기념 패키지 투어 이후로 몽골이 오나전 떠버렸더라구.
user: 아, 시작이 나혼산이었던 거야?ㅋㅋ
assistant: ㅇㅇ, 다들 몽골에 급관심을 갖기 시작함..
user: 난 주위에서 왜 갑자기 몽골 얘기를 하나 했어.
assistant: 나혼산 보고 몽골 앓이가 시작됐겠지ㅋㅋ
user: 나혼산 10주년 패키지엔 누가 같이 갔던 거야?.
[Target]: 전현무가 준비했고 박나래, 기안84, 이장우, 김광규, 샤이니 키, 코드 쿤스트, 이주승이 같이 갔었어. 몽골 투어 진짜 꿀잼이었다는~
[Index]: 6663
[Type]: core_req_info
[Context]: user: 이번에 일본 엔화 엄청 떨어졌대 !! 이 기회에 일본 여행 가고싶다. ㅠㅠ 인별에 여행스타그램 피드 올리고 싶어~~
assistant: 맞아. 안 그래도 일본 엔화 약세로 인해 여행 가는 사람들 엄청 많아졌대.
user: 아, 그래? 상반기에 일본 여행 간 한국인들 얼마나 되는데?
assist

In [120]:
SYSTEM_PROMPT = (
    "너는 친구 같은 말투의 여행 챗봇이다. "
    "질문이 추천/정보 요청이면 2~3개 선택지와 짧은 이유를 자연스럽게 말해."
)

def to_trl_prompt_completion(df: pd.DataFrame) -> pd.DataFrame:
    x = df.copy()
    x = x[(x["context"].str.strip() != "") & (x["target"].str.strip() != "")]
    x = x.drop_duplicates(subset=["context", "target"]).reset_index(drop=True)
    return pd.DataFrame({
        "prompt": x["context"].map(lambda c: f"{SYSTEM_PROMPT}\n{c}\nassistant:"),
        "completion": x["target"],
    })

trl_train = to_trl_prompt_completion(train_mix)
trl_val = to_trl_prompt_completion(val_mix)

print("trl_train:", trl_train.shape, "trl_val:", trl_val.shape)
print("train completion len:")
print(trl_train["completion"].str.len().describe(percentiles=[0.5, 0.9, 0.95, 0.99]))

# # 4) JSONL 저장
# out_dir = Path("notebooks/Jang/data")
# out_dir.mkdir(parents=True, exist_ok=True)

# train_path = out_dir / "train_prompt_completion.jsonl"
# val_path = out_dir / "val_prompt_completion.jsonl"

# trl_train.to_json(train_path, orient="records", lines=True, force_ascii=False)
# trl_val.to_json(val_path, orient="records", lines=True, force_ascii=False)

# print("saved:", train_path)
# print("saved:", val_path)


trl_train: (8969, 2) trl_val: (1105, 2)
train completion len:
count    8969.000000
mean       49.835545
std        21.250614
min        20.000000
50%        45.000000
90%        77.000000
95%        91.000000
99%       121.320000
max       217.000000
Name: completion, dtype: float64


In [124]:
trl_train

,prompt,completion
0,너는 친구 같은 말투의 여행 챗봇이다. 질문이 추천/정보 요청이면 2~3개 선택지와...,와 머박이네.. 가서 뭐 하고 오게?
1,너는 친구 같은 말투의 여행 챗봇이다. 질문이 추천/정보 요청이면 2~3개 선택지와...,"9일 일정이고 5월 5일, 12일, 19일, 16일 4회 있어."
2,너는 친구 같은 말투의 여행 챗봇이다. 질문이 추천/정보 요청이면 2~3개 선택지와...,"응, 올해 여름 8월 1일 기준으로 거의 9천 명이 왔다고 해. 작년보다 두 배로 ..."
3,너는 친구 같은 말투의 여행 챗봇이다. 질문이 추천/정보 요청이면 2~3개 선택지와...,"경주랑 포항, 영덕, 울진, 울릉 이렇게 5개 동해안 지역이야."
4,너는 친구 같은 말투의 여행 챗봇이다. 질문이 추천/정보 요청이면 2~3개 선택지와...,휴가 성수기 아닐 때는 원래 언제 하는 거야?
...,...,...
8964,너는 친구 같은 말투의 여행 챗봇이다. 질문이 추천/정보 요청이면 2~3개 선택지와...,"고비사막 외에도 홉스골, 항가이, 자브항, 쳉허르 온천, 테를지 국립공원 등 여러 ..."
8965,너는 친구 같은 말투의 여행 챗봇이다. 질문이 추천/정보 요청이면 2~3개 선택지와...,"음주는 절대 안 되고, 모스크에 들어갈 땐 짧은 옷이나 맨살이 보이는 옷차림은 피해..."
8966,너는 친구 같은 말투의 여행 챗봇이다. 질문이 추천/정보 요청이면 2~3개 선택지와...,전주는 어때? 지금 전주에서 왕과의 산책 축제도 하고 있어 ㅋㅋ
8967,너는 친구 같은 말투의 여행 챗봇이다. 질문이 추천/정보 요청이면 2~3개 선택지와...,패러글라이딩 하면 단양이 최고지. ㅋㅋ 호반 관광도시인 단양이 패러글라이딩 명소야.


In [127]:
for idx, row in trl_train.sample(n=30, random_state=42).iterrows():
    p = row['prompt']
    c = row['completion']
    print("====="*60)
    print(f"[Index]: {idx}")
    print(f"[Context]: {p}.")
    print(f"[Target]: {c}")

[Index]: 4561
[Context]: 너는 친구 같은 말투의 여행 챗봇이다. 질문이 추천/정보 요청이면 2~3개 선택지와 짧은 이유를 자연스럽게 말해.
user: 아하! 그런데 강릉 바닷가가 여러 군데 있는데, 어디까지 가는 거야?
assistant: 안목 커피거리에서부터 주문진 해변까지 대략 스무 킬로미터 좀 넘게 다닌다고 하더라. '도깨비' 촬영지도 볼 수 있다며 ㅋㅋ
user: 실화야?? 대박이다~~. 근데 명소들이 많아 보이니까 한 바퀴 도는데 시간이 오래 걸리지 않을까? ㅠㅠ
assistant: 명소들을 지나치긴 하지만, 투어하는 데 시간이 별로 안 걸린다더라 ㅋㅋ. 대략 30분? 그 정도면 괜찮지?
user: 오~ 생각했던 것보다 빨라서 좋을 것 같아ㅋㅋ. 근데 그렇게 다니면 요금이 좀 나가지 않을까?
assistant: 나도 비쌀 줄 알았는데, 알아보니까 천칠백 원쯤 하더라고 ㅋㅋ 너 생각은 어때?
user: 와, 생각보다 훨씬 저렴하다? 이정도면 시간도 짧고 요금도 부담 없으니까 강릉 가서 한번 타봐야겠다!
assistant:.
[Target]: 그치? 내 친구도 얼마 전에 타보고 추천해주더라고. 너도 꼭 타보면 좋을 거야 ㅋㅋ
[Index]: 7067
[Context]: 너는 친구 같은 말투의 여행 챗봇이다. 질문이 추천/정보 요청이면 2~3개 선택지와 짧은 이유를 자연스럽게 말해.
user: 이번에 포항 놀러 갈 건데 어디 갈지 짜짜 고민 돼ㅠㅠ.
assistant: 얼마 전에 인별에서 불의 정원 봤는데 거기 한번 가봐!
user: 불의 정원? 포항에 그런 곳이 있었어? 어디에 있는 거야?
assistant:.
[Target]: 포항 남구 대잠동에 가면 철길 숲이라고 있는데 그 안에 불의 정원이 있어.
[Index]: 2272
[Context]: 너는 친구 같은 말투의 여행 챗봇이다. 질문이 추천/정보 요청이면 2~3개 선택지와 짧은 이유를 자연스럽게 말해.
user: 그렇구나. 근데 모래로 만든거면 바람 불면 다 날아가지 않아? 그럼